## Table 1. Characteristics of all studies included in the index and validation datasets.

In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "NMWI")
os.chdir(project_root)

In [2]:
file_path = 'Data/Additional file 1.xlsx'
meta = pd.read_excel(file_path, sheet_name='Table S1', skiprows=1)
meta["Citation"] = meta["Author"].astype(str) + " et al., " + meta["Publication Year"].astype(str)

sra_data_full = pd.read_excel(file_path, sheet_name='Table S2', skiprows=1)
platform = meta[['BioProject Number', 'Sequencing Platform']].drop_duplicates()

sra_data_full = sra_data_full.merge(platform, on='BioProject Number', how='left', validate='many_to_one')

FileNotFoundError: [Errno 2] No such file or directory: 'Data/Additional file 1.xlsx'

In [ ]:
df = sra_data_full.copy()

def fmt_num(x):
    return f"{x:g}"

def age_summary(x):
    x = x.dropna()
    return f"{fmt_num(x.median())} [{fmt_num(x.min())}–{fmt_num(x.max())}]" if len(x) else "NA"

def unique_vals(x):
    return ", ".join(map(str, x.dropna().unique())) or "NA"

def female_summary(x):
    x = x.dropna().astype(str).str.lower()
    n_female = x.isin(['f', 'female']).sum()
    return f"{n_female} ({n_female / len(x) * 100:.1f}%)" if len(x) else "NA"

# Sample counts: training = n, validation = (n), both = n(n)
sample_counts = df.groupby(['BioProject Number', 'Set'])['Sample ID'].nunique().unstack(fill_value=0)
sample_counts['Samples'] = sample_counts.apply(
    lambda x: f"{x.get('Training', 0)}({x.get('Validation', 0)})" if x.get('Training', 0) and x.get('Validation', 0)
    else f"{x.get('Training', 0)}" if x.get('Training', 0)
    else f"({x.get('Validation', 0)})", axis=1
)

# One row per study
table1 = df.groupby('BioProject Number').agg(
    Author=('Author', 'first'),
    Year=('Publication Year', 'first'),
    Phenotype=('Disease', unique_vals),
    Age=('Age', age_summary),
    Female=('Sex', female_summary),
    Specimen=('Collection Technique', unique_vals),
    Site=('Sampling Location', unique_vals),
    Country=('Country', unique_vals),
    Region=('16S Region', unique_vals),
    Sequencing_Platform=('Sequencing Platform', unique_vals)
).reset_index()

table1 = table1.merge(sample_counts[['Samples']], on='BioProject Number', how='left')
table1.insert(0, 'Citation', table1['Author'] + ' et al., ' + table1['Year'].astype(str))

table1 = table1[[
    'Citation', 'Phenotype', 'Samples', 'Age', 'Female', 'Specimen',
    'Sequencing_Platform', 'Site', 'Country', 'Region', 'BioProject Number'
]]

table1.columns = [
    'Citation',
    'Clinical phenotype(s)',
    'Samples in training (validation)',
    'Median age, years [range]',
    'Female, n (%)',
    'Specimen type',
    'Sequencing_Platform',
    'Anatomical site (sampling location)',
    'Country of origin',
    '16S rRNA gene region',
    'NCBI BioProject accession'
]

table1 = table1.sort_values('Citation').reset_index(drop=True)
table1

In [ ]:
# Table 2
df['Health status'] = np.where(
    df['Disease'].astype('string').str.strip().str.lower().eq('healthy'),
    'Healthy', 'Non-healthy'
)
df.loc[df['Disease'].isna(), 'Health status'] = pd.NA

for col in ['Age']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['Sex', 'Sampling Location', '16S Region', 'Sequencing Platform']:
    df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)

d = df[
    df['Set'].isin(['Training', 'Validation']) &
    df['Health status'].notna()
].drop_duplicates(['BioProject Number', 'Sample ID', 'Set', 'Health status']).copy()

groups = [('Training', 'Healthy'), ('Training', 'Non-healthy'),
          ('Validation', 'Healthy'), ('Validation', 'Non-healthy')]

def g(s, h):
    return d[(d['Set'] == s) & (d['Health status'] == h)]

def n_pct(n, total):
    p = 100 * n / total if total else 0
    return f"{n} ({p:.1f}%)"

def study_info(x, var):
    return n_pct(
        x.loc[x[var].notna(), 'BioProject Number'].nunique(),
        x['BioProject Number'].nunique()
    )

def sample_info(x, var):
    return n_pct(x[var].notna().sum(), x['Sample ID'].nunique())

def cat_summary(var, level):
    out = []
    for s, h in groups:
        x = g(s, h)[var].dropna().astype(str).str.strip()
        out.append(n_pct((x == level).sum(), len(x)))
    return out

def male_summary(x):
    x = x.dropna().astype(str).str.lower()
    n_male = x.isin(['m', 'male']).sum()
    return f"{n_male} ({n_male / len(x) * 100:.1f}%)" if len(x) else "NA"

rows = []
def add(label, values):
    rows.append([label, *values])

add('Number of studies', [g(s, h)['BioProject Number'].nunique() for s, h in groups])
add('Number of samples', [g(s, h)['Sample ID'].nunique() for s, h in groups])

add('Age', [''] * 4)
add('  Studies with information, n (%)', [study_info(g(s, h), 'Age') for s, h in groups])
add('  Samples with information, n (%)', [sample_info(g(s, h), 'Age') for s, h in groups])
add('  Median [range]', [age_summary(g(s, h)['Age']) for s, h in groups])

add('Sex', [''] * 4)
add('  Studies with information, n (%)', [study_info(g(s, h), 'Sex') for s, h in groups])
add('  Samples with information, n (%)', [sample_info(g(s, h), 'Sex') for s, h in groups])
add('  Female, n (%)', [female_summary(g(s, h)['Sex']) for s, h in groups])
add('  Male, n (%)', [male_summary(g(s, h)['Sex']) for s, h in groups])


sections = {
    'Sampling location': 'Sampling Location',
    'Hypervariable region': '16S Region',
    'Sequencing Platform': 'Sequencing Platform'
}

for section, var in sections.items():
    levels = d[var].dropna().astype(str).str.strip().drop_duplicates().tolist()
    add(section, [''] * 4)
    add('  Samples with information, n (%)', [sample_info(g(s, h), var) for s, h in groups])
    for level in levels:
        add(f'  {level}, n (%)', cat_summary(var, level))

table2 = pd.DataFrame(rows, columns=[
    'Characteristic',
    'Training | Healthy', 'Training | Non-healthy',
    'Validation | Healthy', 'Validation | Non-healthy'
]).set_index('Characteristic')

table2.columns = pd.MultiIndex.from_tuples([
    ('Training', 'Healthy'), ('Training', 'Non-healthy'),
    ('Validation', 'Healthy'), ('Validation', 'Non-healthy')
])

table2